# Document Loaders

- LangChain은 다양한 파일 형식을 통일된 `Document` 객체로 변환하는 로더를 제공함
- `Document`는 `page_content`(본문 텍스트)와 `metadata`(딕셔너리)로 구성되며, 이후 분할 및 임베딩 단계에서 일관된 인터페이스로 사용됨.

### 참고자료
- https://docs.langchain.com/oss/python/integrations/document_loaders
- https://wikidocs.net/231429

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [1]:
# 필요한 라이브러리 설치
# uv add langchain-openai numpy scikit-learn

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. Document Loaders

### 2.1 TextLoader
- 단일 텍스트 파일

In [3]:
# uv add langchain_community
from langchain_community.document_loaders import TextLoader
from pathlib import Path

hr_policy_path = Path('data2/hr_policy.txt')

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_33224\355349723.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
# encoding='utf-8' : 한글이 깨지지 않도록.
loader = TextLoader(str(hr_policy_path), encoding='utf=8')

# 파일을 읽어서 랭체인 Document 객체 리스트로 변환
documents = loader.load()

In [5]:
print(f"로드된 문서 수: {len(documents)}")
print(f"문서 길이: {len(documents[0].page_content)} 글자")
print(f"메타데이터: {documents[0].metadata}")
print(f"\n미리보기 (처음 300자):")
print(documents[0].page_content[:300])

로드된 문서 수: 1
문서 길이: 3330 글자
메타데이터: {'source': 'data2\\hr_policy.txt'}

미리보기 (처음 300자):
# [인사 정책 매뉴얼] 즐겁고 공정한 직장 문화를 위한 가이드

본 안내서는 우리 회사의 핵심 인사 원칙과 규정을 담고 있습니다. 모든 임직원은 본 정책을 준수하며, 상호 존중과 성과 중심의 문화를 함께 만들어 갑니다.

---

## 1. 휴가 및 근태 정책

우리 회사는 임직원의 충분한 휴식과 일과 삶의 균형(Work-Life Balance)을 전폭적으로 지원합니다.

### 1.1 연차 유급 휴가

* **발생 기준:** * **신입사원:** 입사 1년 미만 시, 1개월 개근 시 1일씩 발생 (최대 11일).
* **정기 연


### 2.2 DirectoryLoader
- 폴더 전체 로드 (다중 포맷)

In [6]:
from langchain_community.document_loaders import DirectoryLoader

In [12]:
company_docs_path = Path("C:\\Users\\playdata2\\Downloads\\company_docs-20260615T033306Z-3-001\\company_docs")
company_docs_path.exists()

True

In [13]:
def load_all_documents(data_dir: Path):
    """폴더에서 .txt와 .md 파일을 모두 로드합니다."""
    all_docs = []
    for pattern in ["**/*.txt", "**/*.md"]:
        loader = DirectoryLoader(
            str(data_dir),
            glob=pattern,
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
        )
        all_docs.extend(loader.load())
    return all_docs

In [14]:
all_docs = load_all_documents(company_docs_path)
print(f"로드된 문서 수: {len(all_docs)}")
for doc in all_docs[:5]:
    print(f"   - {doc.metadata['source']}: {len(doc.page_content)} 글자")

로드된 문서 수: 7
   - C:\Users\playdata2\Downloads\company_docs-20260615T033306Z-3-001\company_docs\hr_policy.txt: 3330 글자
   - C:\Users\playdata2\Downloads\company_docs-20260615T033306Z-3-001\company_docs\dev_standards.md: 5135 글자
   - C:\Users\playdata2\Downloads\company_docs-20260615T033306Z-3-001\company_docs\expense_policy.md: 3304 글자
   - C:\Users\playdata2\Downloads\company_docs-20260615T033306Z-3-001\company_docs\faq.md: 3208 글자
   - C:\Users\playdata2\Downloads\company_docs-20260615T033306Z-3-001\company_docs\onboarding_guide.md: 3304 글자


### 2.3. PyPDF Loader
- PDF 파일 로드
- 이미지+텍스트 페이지 내 텍스트 추출

In [15]:
from langchain_community.document_loaders import PyPDFLoader

In [17]:
filename = Path('data2/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf')
print(filename.resolve())
filename.exists()

C:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\data2\[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf


True

In [18]:
loader = PyPDFLoader(str(filename), mode='page')
pages = loader.load()

print(len(pages))
print(pages[5].metadata)
print(pages[0].page_content)

18
{'producer': 'Hancom PDF 1.3.0.538', 'creator': 'Hancom PDF 1.3.0.538', 'creationdate': '2022-07-29T09:03:16+09:00', 'author': 'kmd kdy', 'moddate': '2022-07-29T09:03:16+09:00', 'pdfversion': '1.4', 'source': 'data2\\[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf', 'total_pages': 18, 'page': 5, 'page_label': '6'}
혁신성장 정책금융 동향 : ICT 산업을 중심으로
  CIS이슈리포트 2022-2호 | 1 |
<요  약>▶혁신성장 정책금융기관*은 혁신성장산업 영위기업을 발굴·지원하기 위한 정책금융 가이드라인**에 따라 혁신성장 기술분야에 대한 금융지원을 강화하고 있음     * 산업은행, 기업은행, 수출입은행, 신용보증기금, 기술보증기금, 중소벤처기업진흥공단, 무역보험공사 등 11개 기관    ** 혁신성장 정책금융 지원 대상을 판단하는 기준으로, ‘9대 테마 – 46개 분야 – 296개 품목’으로 구성￮정책금융기관의 혁신성장 정책금융 공급규모는 2017년 24.1조 원에서 2021년 85.4조 원으로 크게 증가하여 국내 산업 구조의 미래 산업으로의 전환을 충실히 지원하고 있음￮본 보고서는 ICT 산업의 정책금융 지원 트렌드를 파악하고, 혁신성장 정책금융이 집중되는 주요 품목의 기술·시장 동향을 분석함▶혁신성장 ICT 산업은 정보통신(6개 분야, 47개 품목), 전기전자(5개 분야, 27개 품목), 센서측정(3개 분야, 19개 품목) 테마로 구성되며, 혁신성장 정책금융기관의 공급액 규모는 2021년 말 기준 16.9조 원으로 2017년 이후 연평균 39.2% 지속 증가하고 있음￮ICT 산업의 공급액 규모 비중은 혁신성장 정책금융 총 공급 규모의 약 20% 수준임      * (‘17)18.7% → (’18)20.7%  → (’19)18.5

- OCR 기능 활용하여 이미지-텍스트 혼합 페이지 내 텍스트 추출하기

In [ ]:
# #OCR기능 위해 설치
# uv add rapidocr rapidocr-onnxruntime

- 아래 셀은 실행 시 많은 시간이 소요됩니다.

In [19]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

loader = PyPDFLoader(
    filename,
    mode='page',        # PDF를 페이지 단위로 나눠서 Document 객체로 로드
    images_inner_format='markdown-img', # PDF 안의 이미지를 마크다운 이미지 형식으로 본문에 포함
    images_parser=RapidOCRBlobParser()  # 스캔본 PDF나 이미지 기반 표/그림 안의 글자를 추출할 때 사용
)

- 페이지 내 테이블 추출하기

In [20]:
pages = loader.load()
print(pages[5].page_content)

[INFO] 2026-06-15 12:48:55,253 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-15 12:48:55,375 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-15 12:48:55,376 [RapidOCR] main.py:65: Using C:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-15 12:48:55,517 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-15 12:48:55,519 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-15 12:48:55,520 [RapidOCR] main.py:65: Using C:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO]

| 6 | CIS이슈리포트 2022-2호 
▶(주요품목① : 5G 이동통신) 정보통신 테마 내 기술분야 중 혁신성장 정책금융 공급규모가 가장 큰 차세대무선통신미디어 분야의 경우 4G/5G 기술품목의 정책금융 공급 비중이 가장 높은 것으로 확인됨[차세대무선통신미디어 분야 내 기술품목별 혁신성장 정책금융 공급액 추이](단위: 억 원)
▶5G 이동통신 시스템은 ITU(International Telecommunication Union)가 정의한 5세대 이동통신 규격을 만족시키는 무선 이동통신 네트워크 기술로, 2019년부터 국내 서비스를 시작함￮4G 이동통신 시스템(LTE)과 비교할 때 전송속도의 향상(1Gbps→20Gbps), 이동성 향상(350km/h→500km/h에서 끊김없는 데이터 전송 가능), 최대 연결가능 기기수 증가(10만 대 →100만 대 이상), 데이터 전송지연 감소(10ms→1ms) 등의 향상된 기능을 제공함￮5G는 전송속도 향상, 다수기기 접속 및 지연시간 단축을 위해 ①밀리미터파 통신이 가능한 주파수 확장, ②스몰셀(Small cell)을 도입한 기지국, ③다중안테나 송수신(Massive MIMO), ④네트워크 슬라이싱(Network Slicing) 등의 기술을 도입함[5G 주요 요소기술 특징]
자료: 삼정 KPMG



![25,000
20,000
15,000
10,000
RO
5,000
2017
2018
2019
2020
2021
4G/5G
(loT,M2M)
■吾号](#)
![MassiveMMO+
4G5G
（）
4G
2.6Ghz
4G
5G
品号
3.5Ghz
28Ghz
（m）
5G
（1k0H）
loboly
Massive
（km）
loT
5G3.5Ghz28Ghz0
MassiveMIMOEI
舍](#)
![Korea
Credit Information
Services](#)


In [21]:
import textwrap

text = pages[3].page_content

for paragraph in text.split('\n'):
    print(textwrap.fill(paragraph, width=80))
    print()

| 4 | CIS이슈리포트 2022-2호

[혁신성장 ICT 산업 정책금융 공급 현황]
(단위: 억 원, 괄호는 점유율 %)구분2017년 말2018년 말2019년 말2020년 말2021년 말혁신성장 ICT 산업45,075
72,799 81,805 139,687 169,089 (18.7)(20.7)(18.5)(20.3)(19.8)정보통신15,658 27,417
39,033 65,324 77,750 (6.5)(7.8)(8.8)(9.5)(9.1)전기전자26,637 38,521 35,922 62,856
77,485 (11.1)(10.9)(8.1)(9.1)(9.1)센서측정2,780 6,861 6,851 11,506 13,854
(1.2)(1.9)(1.5)(1.7)(1.6)혁신성장 정책금융 총 공급액240,787 351,987 443,180 688,409 854,338
3. 정보통신 테마 혁신성장 정책금융 현황 및 관련 산업 동향▶(지원 현황) 정보통신 테마를 구성하는 기술분야별 정책금융 지원 현황 분석결과,
공급점유율 관점에서는 차세대무선통신미디어 분야에 가장 많은 정책자금이 투입 되고 있으며, 공급량 증가율 관점에서는 능동형컴퓨팅 분야로의 정책자금
지원 증가 속도가 가장 빠른 추세임￮차세대무선통신미디어란 전송속도 향상, 소모전력 절감, 고속이동 중 끊김없는 통신 등 새로운 무선환경에 필요한
통신, 인프라 및 서비스 기술을 통칭하며, 4G/5G/6G, 사물인터넷, 방송통신인프라 등의 품목으로 구성됨-정보통신 테마 내 혁신성장 정책금융
공급 규모의 약 50%를 점유하고 있으며, 이는 초연결 미래사회를 구축하기 위해 네트워크 기반 기술 사업화에 대한 정책자금 공급이 꾸준함에 따른
것으로 분석됨￮능동형컴퓨팅이란 거대하고 복잡해지는 데이터의 효율적 가공과 관리를 위한 인간두뇌와 유사한 형태의 정보처리기술을 말하며, 인공지능,
상황인지컴퓨팅 등의 품목으로 구성됨-컴퓨팅 기술을 활용한 다양한 사업화가 활발히 진행되고 있어 혁신성장 정책금융 공급 규모가 매년 약 100%
수준으로 

### 2.4. PyPDFium2

- 이미지+텍스트 페이지 내 텍스트 추출

In [ ]:
# #PyPDFium2 설치
# uv add pypdfium2

In [22]:
from langchain_community.document_loaders import PyPDFium2Loader

loader2 = PyPDFium2Loader(filename)

data = loader.load()
print(data[5].page_content)

[WARNING] 2026-06-15 14:15:03,324 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-06-15 14:15:39,745 [RapidOCR] main.py:132: The text detection result is empty


| 6 | CIS이슈리포트 2022-2호 
▶(주요품목① : 5G 이동통신) 정보통신 테마 내 기술분야 중 혁신성장 정책금융 공급규모가 가장 큰 차세대무선통신미디어 분야의 경우 4G/5G 기술품목의 정책금융 공급 비중이 가장 높은 것으로 확인됨[차세대무선통신미디어 분야 내 기술품목별 혁신성장 정책금융 공급액 추이](단위: 억 원)
▶5G 이동통신 시스템은 ITU(International Telecommunication Union)가 정의한 5세대 이동통신 규격을 만족시키는 무선 이동통신 네트워크 기술로, 2019년부터 국내 서비스를 시작함￮4G 이동통신 시스템(LTE)과 비교할 때 전송속도의 향상(1Gbps→20Gbps), 이동성 향상(350km/h→500km/h에서 끊김없는 데이터 전송 가능), 최대 연결가능 기기수 증가(10만 대 →100만 대 이상), 데이터 전송지연 감소(10ms→1ms) 등의 향상된 기능을 제공함￮5G는 전송속도 향상, 다수기기 접속 및 지연시간 단축을 위해 ①밀리미터파 통신이 가능한 주파수 확장, ②스몰셀(Small cell)을 도입한 기지국, ③다중안테나 송수신(Massive MIMO), ④네트워크 슬라이싱(Network Slicing) 등의 기술을 도입함[5G 주요 요소기술 특징]
자료: 삼정 KPMG



![25,000
20,000
15,000
10,000
RO
5,000
2017
2018
2019
2020
2021
4G/5G
(loT,M2M)
■吾号](#)
![MassiveMMO+
4G5G
（）
4G
2.6Ghz
4G
5G
品号
3.5Ghz
28Ghz
（m）
5G
（1k0H）
loboly
Massive
（km）
loT
5G3.5Ghz28Ghz0
MassiveMIMOEI
舍](#)
![Korea
Credit Information
Services](#)


- 페이지 내 테이블 추출하기

In [23]:
print(data[3].page_content)

| 4 | CIS이슈리포트 2022-2호 
[혁신성장 ICT 산업 정책금융 공급 현황]                                                            (단위: 억 원, 괄호는 점유율 %)구분2017년 말2018년 말2019년 말2020년 말2021년 말혁신성장 ICT 산업45,075 72,799 81,805 139,687 169,089 (18.7)(20.7)(18.5)(20.3)(19.8)정보통신15,658 27,417 39,033 65,324 77,750 (6.5)(7.8)(8.8)(9.5)(9.1)전기전자26,637 38,521 35,922 62,856 77,485 (11.1)(10.9)(8.1)(9.1)(9.1)센서측정2,780 6,861 6,851 11,506 13,854 (1.2)(1.9)(1.5)(1.7)(1.6)혁신성장 정책금융 총 공급액240,787 351,987 443,180 688,409 854,338 3. 정보통신 테마 혁신성장 정책금융 현황 및 관련 산업 동향▶(지원 현황) 정보통신 테마를 구성하는 기술분야별 정책금융 지원 현황 분석결과, 공급점유율 관점에서는 차세대무선통신미디어 분야에 가장 많은 정책자금이 투입 되고 있으며, 공급량 증가율 관점에서는 능동형컴퓨팅 분야로의 정책자금 지원 증가 속도가 가장 빠른 추세임￮차세대무선통신미디어란 전송속도 향상, 소모전력 절감, 고속이동 중 끊김없는 통신 등 새로운 무선환경에 필요한 통신, 인프라 및 서비스 기술을 통칭하며, 4G/5G/6G, 사물인터넷, 방송통신인프라 등의 품목으로 구성됨-정보통신 테마 내 혁신성장 정책금융 공급 규모의 약 50%를 점유하고 있으며, 이는 초연결 미래사회를 구축하기 위해 네트워크 기반 기술 사업화에 대한 정책자금 공급이 꾸준함에 따른 것으로 분석됨￮능동형컴퓨팅이란 거대하고 복잡해지는 데이터의 효율적 가공과 관리를 위한 인간두뇌와 유사한 형태의 정보처리기술을 말하며, 인공지능, 상황인지컴퓨팅 등의 품목으로 구성됨-컴퓨팅 기술

### 2.5. PyPDFLoader vs PyPDFium2Loader

- PyPDFLoader의 텍스트 추출 소요 시간

In [24]:
%%time

loader = PyPDFLoader(filename, mode="page")

pages = loader.load()

CPU times: total: 2.03 s
Wall time: 2.05 s


- PyPDFium2의 텍스트 추출 소요 시간

In [25]:
%%time

loader = PyPDFium2Loader(filename)

pages = loader.load()

CPU times: total: 219 ms
Wall time: 261 ms
